# XManager with MaxText on Vertex Training Cluster

This notebook demonstrates how to use XManager to launch and manage **MaxText (JAX) distributed training jobs** on Slurm-based Vertex Training Clusters.

## What You'll Learn

1. **Setup** - Initialize XManager in a notebook environment
2. **Configuration** - Set cluster connection and job parameters
3. **Cluster Configs** - Pre-configured NCCL settings for different GPU types
4. **MaxText Training** - Launch JAX-based distributed training with `setup_jax_coordinator`
5. **Experiment Tracking** - List, query, and monitor experiments
6. **Vizier Hyperparameter Optimization** - Multi-objective Bayesian optimization
7. **Vizier Results & Monitoring** - Track trials, view logs, cancel jobs

## Supported Cluster Types

| Cluster Type | GPU | GPUs/Node | Network |
|-------------|-----|-----------|----------|
| `hcc-a3m` | H100 | 8 | TCPXO |
| `hcc-a3u` | H200 | 8 | gIB |
| `hcc-a4` | B200 | 8 | gIB |
| `hcc-a3h` | H100 | 8 | gIB |

---
## 1. Setup and Installation

Install dependencies and initialize XManager for notebook use.

In [ ]:
# Install dependencies
!pip install nest_asyncio pandas -q

# For development, install from local source
# !pip install -e .

In [ ]:
from xmanager import xm
from xmanager import xm_local

import pandas as pd
from IPython.display import display, HTML

# Initialize absl flags (required for XManager in notebooks)
import sys
from absl import flags

# Parse flags with empty argv to initialize XManager
# This is needed because XManager uses absl flags internally
if not flags.FLAGS.is_parsed():
    flags.FLAGS(sys.argv[:1])  # Only pass the script name, ignore notebook args

# Enable nested asyncio for Jupyter compatibility
# XManager uses asyncio internally, which conflicts with Jupyter's event loop
import nest_asyncio
nest_asyncio.apply()

print("XManager initialized successfully!")

---
## 2. Configuration

Set your cluster connection details and job parameters.

### Key Concepts:
- **work_dir**: Directory on cluster containing your scripts. Mounted at `/mnt/jobs` inside the container.
- **container_image**: Path to a squashfs container image (.sqsh) on the cluster
- **config_file**: MaxText YAML config file (path inside the container)
- **GCS bucket**: Used for `base_output_directory` (checkpoints) and TensorBoard logs

In [ ]:
# Cluster connection - update these for your environment

# SSH connection to cluster login node
LOGIN_NODE = "vmdsa3u04-login-001"  # Cluster login node name
USE_GCLOUD_SSH = True  # Use gcloud compute ssh (vs direct ssh)
SSH_HOSTNAME = "nic0.vmdsa3u04-login-001.europe-west4-a.c.ai-infra-recipe-validation.internal.gcpnode.com"

# Cluster settings

CLUSTER_TYPE = "hcc-a3u"  # Options: hcc-a3m, hcc-a3u, hcc-a4, hcc-a3h
PARTITION = "a3u"         # Slurm partition
ACCOUNT = None            # Slurm account (optional)
NUM_NODES = 2             # Number of nodes

# Paths on cluster

# Work directory - contains training scripts, mounted at /mnt/jobs in container
WORK_DIR = "/home/abhishekbhgwt_google_com/vertexai-mds/nemo"

# Container image (squashfs format)
CONTAINER_IMAGE = "/mnt/lustre/ls1-europe-west4-a/images/jax-maxtext-2025-10-01.sqsh"

# MaxText config file (path inside the container, relative to /mnt/jobs mount)
CONFIG_FILE = "/mnt/jobs/config.yaml"

# GCS bucket for base_output_directory (checkpoints, metrics)
GCS_BUCKET = "ai-infra-gcs-europe-west4"

# Local files to rsync to work_dir on the cluster before job submission
LOCAL_FILES = [
    # "config.yaml",
    # "my_script.py",
]

# TensorBoard settings

# GCP project for TensorBoard
TENSORBOARD_PROJECT = "ai-infra-recipe-validation"

# Vertex AI TensorBoard instance display name
TENSORBOARD_NAME = "ai-infra-europe-west4-tb"

# GCS bucket path for TensorBoard logs (bucket-name format, no gs:// or /gcs/)
TENSORBOARD_GCS_PATH = "ai-infra-gcs-europe-west4"

# Vertex AI region for TensorBoard
TENSORBOARD_REGION = "europe-west4"

# Model / training

MODEL = "llama3.1-70b"  # Model name (MaxText model_name config key)
STEPS = 10               # Number of training steps

print("Configuration loaded!")
print(f"  Cluster: {CLUSTER_TYPE}")
print(f"  Partition: {PARTITION}")
print(f"  Nodes: {NUM_NODES}")
print(f"  Container: {CONTAINER_IMAGE}")
print(f"  Work dir: {WORK_DIR}")
print(f"  Config file: {CONFIG_FILE}")
print(f"  GCS bucket: gs://{GCS_BUCKET}")
print(f"  Local files: {LOCAL_FILES}")
print(f"  Model: {MODEL}")
print(f"  Steps: {STEPS}")
print(f"  TensorBoard: {TENSORBOARD_NAME} ({TENSORBOARD_REGION})")

---
## 3. Cluster Configuration Details

Each cluster type has **pre-configured NCCL settings** optimized for its networking hardware.

XManager automatically applies these settings when you specify `cluster_type`.

In [ ]:
# Create a minimal executor to inspect cluster config
executor = xm_local.VertexTrainingCluster(
    cluster_type=CLUSTER_TYPE,
    partition=PARTITION,
)

# Get the cluster configuration
cluster_config = executor.get_cluster_config()

print(f"Cluster Configuration for {CLUSTER_TYPE}")
print("=" * 50)
print(f"GPU Type: {cluster_config.gpu_type}")
print(f"GPUs per Node: {cluster_config.gpus_per_node}")
print(f"Total GPUs (with {NUM_NODES} nodes): {NUM_NODES * cluster_config.gpus_per_node}")
print()
print("NCCL Environment Variables (auto-configured):")
for key, value in cluster_config.nccl_env_vars.items():
    print(f"  {key}={value}")

---
## 4. MaxText Training Job

Launch a distributed MaxText (JAX) training job.

### How it works:

1. **Executor Setup**: Configure cluster, container, JAX coordinator, environment
2. **Job Creation**: Specify `python` with MaxText `train.py` and `key=value` config overrides
3. **Sbatch Generation**: XManager generates an sbatch script with:
   - JAX coordinator setup via `setup_jax_coordinator=True`
   - NCCL + XLA configuration for the cluster type
   - Container execution via `srun --container-image`
4. **Submission**: Script is submitted via SSH to the login node

Use `xm.ShellSafeArg` for arguments containing shell variables (like `${AIP_TENSORBOARD_LOG_DIR}`) that should expand at runtime, not be quoted.

In [ ]:
# Container mounts:
# - work_dir:/mnt/jobs — training scripts and config accessible inside container
# - /gcs:/gcs — GCS FUSE mount for reading/writing GCS buckets
# Note: gIB mount (/usr/local/gib) is automatically added by XManager
container_mounts = [
    f"{WORK_DIR}:/mnt/jobs",
    "/gcs:/gcs",
]

# Environment variables for JAX/XLA training
env_vars = {
    'NCCL_SOCKET_IFNAME': 'enp0s19,enp192s20',
    'NCCL_DEBUG': 'VERSION',
    'CUDA_DEVICE_MAX_CONNECTIONS': '1',
    'JAX_PLATFORMS': 'cuda',
    'JAX_REMOVE_CUSTOM_PARTITIONING_PTR_FROM_CACHE_KEY': 'true',
    'JAX_ENABLE_PGLE': 'false',
    'SLURM_NTASKS_PER_NODE': '8',
    'TF_CPP_MIN_LOG_LEVEL': '0',
    'XLA_PYTHON_CLIENT_MEM_FRACTION': '0.98',
    'NVTE_FUSED_ATTN': '1',
    'XLA_FLAGS': (
        '--xla_gpu_enable_latency_hiding_scheduler=true '
        '--xla_gpu_enable_triton_gemm=false '
        '--xla_gpu_enable_command_buffer=FUSION,CUSTOM_CALL '
        '--xla_gpu_all_reduce_combine_threshold_bytes=2147483648 '
        '--xla_gpu_all_gather_combine_threshold_bytes=2147483648 '
        '--xla_gpu_reduce_scatter_combine_threshold_bytes=16777216 '
        '--xla_gpu_enable_pipelined_all_gather=true '
        '--xla_gpu_enable_pipelined_reduce_scatter=true '
        '--xla_gpu_enable_pipelined_all_reduce=true '
        '--xla_gpu_enable_while_loop_double_buffering=true '
        '--xla_gpu_enable_all_gather_combine_by_dim=false '
        '--xla_gpu_enable_reduce_scatter_combine_by_dim=false '
        '--xla_disable_hlo_passes=rematerialization'
    ),
}

# Environment variables to pass through to container via --container-env
container_env_passthrough = [
    'JAX_COORDINATOR_ADDRESS',
    'JAX_PLATFORMS',
    'NCCL_DEBUG',
    'XLA_FLAGS',
    'SLURM_NTASKS_PER_NODE',
]

# TensorBoard integration (optional) - enables native VTC TensorBoard support
# VMDS will set AIP_TENSORBOARD_LOG_DIR inside the job
tensorboard = xm_local.TensorboardCapability(
    name=TENSORBOARD_NAME,
    base_output_directory=TENSORBOARD_GCS_PATH,
)

# Prologue commands run inside the container before the training command.
# AIP_TENSORBOARD_LOG_DIR is set by VMDS as /gcs/... but MaxText expects gs://...
# so we convert the path format.
prologue_commands = [
    'echo "Starting MaxText training"',
    'echo "AIP_TENSORBOARD_LOG_DIR=${AIP_TENSORBOARD_LOG_DIR}"',
    'export AIP_TENSORBOARD_LOG_DIR="${AIP_TENSORBOARD_LOG_DIR/#\\/gcs\\//gs:\\/\\/}"',
    'echo "AIP_TENSORBOARD_LOG_DIR (converted): ${AIP_TENSORBOARD_LOG_DIR}"',
]

# Create the executor
executor = xm_local.VertexTrainingCluster(
    # Cluster settings
    cluster_type=CLUSTER_TYPE,
    partition=PARTITION,
    account=ACCOUNT,

    # Resource requirements (replicas = number of nodes)
    requirements=xm.JobRequirements(replicas=NUM_NODES),

    # SSH connection to login node
    login_node=LOGIN_NODE,
    use_gcloud_ssh=USE_GCLOUD_SSH,
    ssh_hostname=SSH_HOSTNAME,

    # Working directory
    work_dir=WORK_DIR,

    # Container configuration
    container_image=CONTAINER_IMAGE,
    container_mounts=container_mounts,
    container_env_passthrough=container_env_passthrough,

    # JAX coordinator (replaces use_mpi for JAX workloads)
    setup_jax_coordinator=True,

    # Environment variables
    env_vars=env_vars,

    # Prologue commands (AIP_TENSORBOARD_LOG_DIR path conversion)
    prologue_commands=prologue_commands,

    # Stream output to console
    stream_output=True,

    # Local files to rsync to work_dir on the cluster
    local_files=LOCAL_FILES,

    # TensorBoard integration
    tensorboard=tensorboard,
    tensorboard_region=TENSORBOARD_REGION,
    tensorboard_project=TENSORBOARD_PROJECT,
)

print("Executor configured!")
print(f"  Container: {CONTAINER_IMAGE}")
print(f"  Mounts: {container_mounts}")
print(f"  JAX coordinator: enabled")
print(f"  Local files: {LOCAL_FILES}")
print(f"  TensorBoard: {TENSORBOARD_NAME} ({TENSORBOARD_REGION})")

In [ ]:
# MaxText takes positional args: python train.py config.yaml key=value key=value ...
# CLI overrides take precedence over the YAML config file.
#
# The sbatch template exports these variables:
# - SLURM_JOB_ID: Slurm job identifier
# - AIP_TENSORBOARD_LOG_DIR: (VMDS sets this) GCS path for TensorBoard logs
#
# Use ShellSafeArg for arguments containing shell variables.
# This prevents them from being quoted (which would prevent expansion).

maxtext_args = [
    f'steps={STEPS}',
    f'base_output_directory=gs://{GCS_BUCKET}',
    'tensorboard_dir=${AIP_TENSORBOARD_LOG_DIR}',
    'run_name=job-${SLURM_JOB_ID}',
    'packing=False',
    f'model_name={MODEL}',
]

# Build the full argument list:
# python src/MaxText/train.py /mnt/jobs/config.yaml key=value ...
training_args = [
    'src/MaxText/train.py',
    CONFIG_FILE,
] + [xm.ShellSafeArg(a) for a in maxtext_args]

print("MaxText training arguments:")
for arg in training_args:
    if isinstance(arg, xm.ShellSafeArg):
        print(f"  {arg.arg}")
    else:
        print(f"  {arg}")

In [ ]:
import time
timestamp = time.strftime("%Y%m%d-%H%M%S")

with xm_local.create_experiment(experiment_title=f"maxtext_train_{timestamp}") as experiment:

    # Create job with python as executable
    job = xm.Job(
        executable=xm.Binary(path='python'),
        args=training_args,
        executor=executor,
    )

    # Submit job
    print("Submitting job to Slurm...")
    experiment.add(xm.JobGroup(job=job))

    experiment_id = experiment.experiment_id
    print(f"\nExperiment ID: {experiment_id}")
    print(f"\nMonitor with:")
    print(f"  squeue --me")
    print(f"  xmanager list")
    print(f"\nLogs will be in: {WORK_DIR}/slurm-<job_id>.out")

---
## 5. Experiment Tracking & Monitoring

XManager tracks all experiments in a **local SQLite database**.

You can list, query, and retrieve experiment details programmatically or via CLI.

In [ ]:
# Helper function to run commands on the cluster
import subprocess

def run_on_cluster(cmd):
    """Run a command on the cluster via SSH."""
    if USE_GCLOUD_SSH:
        ssh_cmd = [
            "gcloud", "compute", "ssh", LOGIN_NODE,
            "--", "-T",
            "-o", f"Hostname={SSH_HOSTNAME}",
            cmd
        ]
    else:
        ssh_cmd = ["ssh", LOGIN_NODE, cmd]

    result = subprocess.run(ssh_cmd, capture_output=True, text=True)
    return result.stdout, result.stderr

print("SSH helper function defined.")

In [ ]:
# List all experiments
experiments = xm_local.list_experiments()

# Convert to DataFrame for nice display
exp_data = []
for exp in experiments[-10:]:  # Last 10 experiments
    exp_data.append({
        "ID": exp.experiment_id,
        "Title": exp._experiment_title,
    })

df = pd.DataFrame(exp_data)
print("Recent Experiments:")
display(df)

In [ ]:
# Get a specific experiment by ID
# Replace with an actual experiment ID from your list
EXPERIMENT_ID = experiment_id  # Uses the ID from the job submitted above

if EXPERIMENT_ID:
    exp = xm_local.get_experiment(EXPERIMENT_ID)
    print(f"Experiment: {exp._experiment_title}")
    print(f"ID: {exp.experiment_id}")

    work_units = exp._experiment_units
    print(f"\nWork Units: {len(work_units)}")
    for wu in work_units:
        print(f"  - Work Unit {wu.work_unit_id}")
        if hasattr(wu, '_non_local_execution_handles'):
            for handle in wu._non_local_execution_handles:
                if hasattr(handle, 'slurm_job_id'):
                    print(f"    Slurm Job ID: {handle.slurm_job_id}")
                if hasattr(handle, 'get_status'):
                    try:
                        status = handle.get_status()
                        status_name = status._status.name if hasattr(status, '_status') else str(status)
                        message = status.message if hasattr(status, 'message') and status.message else ""
                        print(f"    Status: {status_name}")
                        if message:
                            print(f"    Message: {message}")
                    except Exception as e:
                        print(f"    Status check failed: {e}")
else:
    print("No experiments found. Run a job first!")

In [ ]:
# Check running jobs on the cluster
stdout, stderr = run_on_cluster("squeue --me")
print("Running Jobs (squeue --me):")
print(stdout if stdout.strip() else "No running jobs")

In [ ]:
# View logs for a specific Slurm job
SLURM_JOB_ID = "4"  # Replace with actual job ID

# View last 30 lines of log
stdout, _ = run_on_cluster(f"tail -30 {WORK_DIR}/slurm-{SLURM_JOB_ID}.out 2>/dev/null || echo 'Log file not found'")
print(f"Log tail (slurm-{SLURM_JOB_ID}.out):")
print(stdout)

---
## 6. Vizier Hyperparameter Optimization

Use **Vertex AI Vizier** for intelligent multi-objective hyperparameter optimization.

### How it works:

1. **Study Definition**: Define parameters and metrics to optimize
2. **Trial Suggestion**: Vizier uses Bayesian Optimization to suggest hyperparameters
3. **Job Execution**: XManager runs VTC jobs with suggested hyperparameters
4. **Metric Reading**: XManager reads metrics from GCS TensorBoard logs
5. **Optimization**: Vizier uses results to suggest better hyperparameters

### Multi-Objective Optimization:
- **Minimize** `learning/loss` — lower loss means better convergence
- **Maximize** `perf/per_device_tflops_per_sec` — higher throughput means better efficiency

In [ ]:
from google.cloud import aiplatform_v1beta1 as aip
from xmanager.vizier import vizier_cloud

def get_study_spec():
    """Define the Vizier study specification.

    Parameters:
    - learning_rate: log-scale double in [1e-5, 1e-3]
    - per_device_batch_size: discrete values {1, 2, 4}

    Objectives:
    - Minimize learning/loss
    - Maximize perf/per_device_tflops_per_sec
    """
    return aip.StudySpec(
        algorithm=aip.StudySpec.Algorithm.ALGORITHM_UNSPECIFIED,
        parameters=[
            aip.StudySpec.ParameterSpec(
                parameter_id='learning_rate',
                double_value_spec=aip.StudySpec.ParameterSpec.DoubleValueSpec(
                    min_value=1e-5,
                    max_value=1e-3,
                ),
                scale_type=aip.StudySpec.ParameterSpec.ScaleType.UNIT_LOG_SCALE,
            ),
            aip.StudySpec.ParameterSpec(
                parameter_id='per_device_batch_size',
                discrete_value_spec=aip.StudySpec.ParameterSpec.DiscreteValueSpec(
                    values=[1, 2, 4],
                ),
            ),
        ],
        metrics=[
            aip.StudySpec.MetricSpec(
                metric_id='learning/loss',
                goal=aip.StudySpec.MetricSpec.GoalType.MINIMIZE,
            ),
            aip.StudySpec.MetricSpec(
                metric_id='perf/per_device_tflops_per_sec',
                goal=aip.StudySpec.MetricSpec.GoalType.MAXIMIZE,
            ),
        ],
    )

print("Vizier study spec defined!")
print("Parameters:")
print("  - learning_rate: double [1e-5, 1e-3] (log scale)")
print("  - per_device_batch_size: discrete {1, 2, 4}")
print("Objectives:")
print("  - Minimize learning/loss")
print("  - Maximize perf/per_device_tflops_per_sec")

In [ ]:
# Base MaxText arguments for Vizier. Vizier will inject hyperparameters
# (learning_rate, per_device_batch_size) as additional key=value CLI overrides.

vizier_maxtext_args = [
    f'steps={STEPS}',
    f'base_output_directory=gs://{GCS_BUCKET}',
    'tensorboard_dir=${AIP_TENSORBOARD_LOG_DIR}',
    'run_name=job-${SLURM_JOB_ID}',
    'packing=False',
]

vizier_training_args = [
    'src/MaxText/train.py',
    CONFIG_FILE,
] + [xm.ShellSafeArg(a) for a in vizier_maxtext_args]

# Create base job (Vizier will merge in hyperparams as additional args)
vizier_job = xm.Job(
    executable=xm.Binary(path='python'),
    args=vizier_training_args,
    executor=executor,  # Uses the executor defined in section 4
)

print("Base job for Vizier exploration created!")
print("Vizier will inject: learning_rate=X per_device_batch_size=Y")

In [ ]:
import asyncio

# Vizier optimization settings
NUM_TRIALS = 5   # Total number of hyperparameter combinations to try
NUM_PARALLEL = 1  # Run 1 trial at a time

# Metric names to collect from TensorBoard logs
METRIC_NAMES = ['learning/loss', 'perf/per_device_tflops_per_sec']

# Vizier project/region
VIZIER_PROJECT = "ai-infra-recipe-validation"
VIZIER_REGION = "europe-west4"

timestamp = time.strftime("%Y%m%d-%H%M%S")

async def run_vizier_exploration():
    async with xm_local.create_experiment(experiment_title=f"maxtext_vizier_{timestamp}") as experiment:

        print(f"Starting Vizier exploration...")
        print(f"  Trials: {NUM_TRIALS}")
        print(f"  Parallel: {NUM_PARALLEL}")
        print(f"  Metrics: {METRIC_NAMES}")
        print(f"  GCS base: gs://{GCS_BUCKET}")
        print()

        exploration = vizier_cloud.VizierExploration(
            experiment=experiment,
            job=vizier_job,
            study_factory=vizier_cloud.NewStudy(
                study_config=get_study_spec(),
                project=VIZIER_PROJECT,
                location=VIZIER_REGION,
            ),
            num_trials_total=NUM_TRIALS,
            num_parallel_trial_runs=NUM_PARALLEL,
            metric_names=METRIC_NAMES,
            gcs_log_base=f'gs://{GCS_BUCKET}',
        )

        print("Launching Vizier study...")
        print("Monitor jobs: squeue --me")
        exploration.launch(poll_frequency_in_sec=60)

        vizier_experiment_id = experiment.experiment_id
        print(f"\nVizier exploration completed!")
        print(f"Experiment ID: {vizier_experiment_id}")

# Run the exploration (blocks until all trials complete)
asyncio.get_event_loop().run_until_complete(run_vizier_exploration())

---
## 7. Vizier Results & Monitoring

Track Vizier experiment results after optimization completes.

In [ ]:
# Check Vizier experiment — list all trial work units and their statuses
VIZIER_EXPERIMENT_ID = vizier_experiment_id  # From the Vizier exploration above

if VIZIER_EXPERIMENT_ID:
    exp = xm_local.get_experiment(VIZIER_EXPERIMENT_ID)
    print(f"Experiment: {exp._experiment_title}")
    print(f"ID: {exp.experiment_id}")

    work_units = exp._experiment_units
    print(f"\nTrials (Work Units): {len(work_units)}")
    for wu in work_units:
        print(f"  - Work Unit {wu.work_unit_id}")
        if hasattr(wu, '_non_local_execution_handles'):
            for handle in wu._non_local_execution_handles:
                if hasattr(handle, 'slurm_job_id'):
                    print(f"    Slurm Job ID: {handle.slurm_job_id}")
                if hasattr(handle, 'get_status'):
                    try:
                        status = handle.get_status()
                        status_name = status._status.name if hasattr(status, '_status') else str(status)
                        print(f"    Status: {status_name}")
                    except Exception as e:
                        print(f"    Status check failed: {e}")
else:
    print("No Vizier experiment ID. Run the Vizier exploration first!")

In [ ]:
# View logs for a specific Vizier trial
TRIAL_SLURM_JOB_ID = "4"  # Replace with actual Slurm job ID from above

stdout, _ = run_on_cluster(f"tail -30 {WORK_DIR}/slurm-{TRIAL_SLURM_JOB_ID}.out 2>/dev/null || echo 'Log file not found'")
print(f"Trial log tail (slurm-{TRIAL_SLURM_JOB_ID}.out):")
print(stdout)

In [ ]:
# Cancel jobs (uncomment to use)
SLURM_JOB_ID_TO_CANCEL = "999"  # Replace with job ID to cancel

# Uncomment to actually cancel:
# stdout, stderr = run_on_cluster(f"scancel {SLURM_JOB_ID_TO_CANCEL}")
# print(f"Cancelled job {SLURM_JOB_ID_TO_CANCEL}")

print(f"To cancel job {SLURM_JOB_ID_TO_CANCEL}:")
print(f"  Uncomment the lines above, or run:")
print(f"  scancel {SLURM_JOB_ID_TO_CANCEL}")